# 面试题：分类模型的概率校准和业务阈值应该怎样实现与评估？

AUC 高不代表 `0.8` 真有 80% 命中率，也不代表阈值 0.5 符合业务成本。本 Notebook 手写时间切分、逻辑模型、NLL/Brier/ECE、reliability bins、temperature scaling、成本阈值、拒绝区间、切片审计和发布 wrapper。

使用 PyTorch 基础层与 NumPy 统计，不调用 sklearn calibration/metrics。

In [ ]:
import copy,hashlib,json,math,random,warnings
from types import MappingProxyType
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
SEED77=7701
np.random.seed(SEED77); torch.manual_seed(SEED77); torch.set_num_threads(1)
def canonical77(x): return json.dumps(x,sort_keys=True,separators=(",",":"))
def sha77(x): return hashlib.sha256(x).hexdigest()
assert torch.get_num_threads()==1

## 1. train/calibration/threshold/test 四段时间切分

训练模型、拟合温度、选业务阈值和最终测试必须用不同时间段，否则后两步会过拟合测试。生成 4,000 条时序样本；后期有轻微 intercept drift，用来展示校准也会过期。

group 表示两个业务来源，后续检查切片 calibration。label 只由潜在概率采样，不作为特征。

In [ ]:
rng77=np.random.default_rng(SEED77); n77=4000; x77=rng77.normal(size=(n77,3)).astype(np.float32); group77=(rng77.random(n77)<.35).astype(np.int64)
drift77=np.linspace(0,.6,n77); true_logit77=1.4*x77[:,0]-.8*x77[:,1]+.5*x77[:,2]+.35*group77+drift77-0.4; true_prob77=1/(1+np.exp(-true_logit77)); y77=(rng77.random(n77)<true_prob77).astype(np.float32)
splits77={"train":np.arange(0,2200),"cal":np.arange(2200,2900),"threshold":np.arange(2900,3400),"test":np.arange(3400,4000)}
assert sum(len(v) for v in splits77.values())==n77 and all(bool((np.diff(v)>0).all()) for v in splits77.values())
assert max(splits77["train"])<min(splits77["cal"])<min(splits77["threshold"])<min(splits77["test"])
assert 0<y77.mean()<1 and x77.shape==(4000,3)

## 2. 训练一个故意过置信的分类器

先训练线性 logistic model，再把 logits 乘 1.8 模拟蒸馏/过训练造成的过置信。缩放不改变 score 排序，因此 AUC 不变，却会恶化概率解释。

模型只在 train 优化；cal/test 完全 no-grad。

In [ ]:
class Logistic77(nn.Module):
    def __init__(self): super().__init__(); self.linear=nn.Linear(3,1)
    def forward(self,x):
        if x.ndim!=2 or x.shape[1]!=3 or not torch.isfinite(x).all(): raise ValueError("classifier_input")
        return self.linear(x).squeeze(-1)
tx77=torch.tensor(x77); ty77=torch.tensor(y77); model77=Logistic77(); opt77=torch.optim.Adam(model77.parameters(),lr=.05)
for _ in range(180):
    idx=splits77["train"]; loss=F.binary_cross_entropy_with_logits(model77(tx77[idx]),ty77[idx]); opt77.zero_grad(set_to_none=True); loss.backward(); opt77.step()
with torch.no_grad(): raw_logits77=model77(tx77)*1.8
assert raw_logits77.shape==(n77,) and torch.isfinite(raw_logits77).all()
assert any(p.grad is not None for p in model77.parameters()) and float(loss)<.7
try: model77(torch.zeros(2,2)); raise AssertionError("wrong input accepted")
except ValueError as e: assert str(e)=="classifier_input"

## 3. NLL、Brier 与 ECE

NLL 对错误高置信惩罚强；Brier 是概率平方误差；ECE 按置信区间比较平均 confidence 与 accuracy。ECE 依赖 bin 数和样本量，不能单独作为 proper scoring rule。

二分类 reliability 使用 positive probability，而不是 `max(p,1-p)`；这里还返回每个 bin 的 count 便于发现空桶。

In [ ]:
def calibration_metrics77(logits,labels,bins=10):
    logits=torch.as_tensor(logits,dtype=torch.float32); labels=torch.as_tensor(labels,dtype=torch.float32)
    if logits.shape!=labels.shape or logits.ndim!=1 or bins<2 or not torch.isfinite(logits).all(): raise ValueError("calibration_metric_contract")
    p=torch.sigmoid(logits); nll=float(F.binary_cross_entropy(p,labels)); brier=float(((p-labels)**2).mean()); edges=torch.linspace(0,1,bins+1); ece=0.; rows=[]
    for i in range(bins):
        mask=(p>=edges[i])&((p<edges[i+1]) if i<bins-1 else (p<=edges[i+1])); count=int(mask.sum())
        conf=float(p[mask].mean()) if count else float("nan"); acc=float(labels[mask].mean()) if count else float("nan")
        if count: ece+=count/len(p)*abs(conf-acc)
        rows.append((float(edges[i]),float(edges[i+1]),count,conf,acc))
    return {"nll":nll,"brier":brier,"ece":ece,"bins":rows}
test_idx77=splits77["test"]; raw_metric77=calibration_metrics77(raw_logits77[test_idx77],ty77[test_idx77])
assert all(math.isfinite(raw_metric77[k]) for k in ("nll","brier","ece")) and len(raw_metric77["bins"])==10
assert sum(r[2] for r in raw_metric77["bins"])==len(test_idx77) and 0<=raw_metric77["ece"]<=1
try: calibration_metrics77(torch.zeros(2),torch.zeros(3)); raise AssertionError("metric shape accepted")
except ValueError as e: assert str(e)=="calibration_metric_contract"

## 4. Temperature scaling

拟合单个正温度 `T=exp(log_T)`，校准 logit=`z/T`，只在 calibration split 最小化 NLL。`T>1` 会减弱过置信；单调缩放保持排序不变。

温度不能修复所有 group/非线性 miscalibration，也不能使用 test 标签拟合。

In [ ]:
log_t77=torch.tensor(0.,requires_grad=True); opt_t77=torch.optim.LBFGS([log_t77],lr=.2,max_iter=60,line_search_fn="strong_wolfe"); cal_idx77=splits77["cal"]
before_cal_nll77=float(F.binary_cross_entropy_with_logits(raw_logits77[cal_idx77],ty77[cal_idx77]))
def closure77():
    opt_t77.zero_grad(); value=F.binary_cross_entropy_with_logits(raw_logits77[cal_idx77]/log_t77.exp(),ty77[cal_idx77]); value.backward(); return value
opt_t77.step(closure77); temperature77=float(log_t77.exp().detach()); calibrated_logits77=raw_logits77/temperature77
after_cal_nll77=float(F.binary_cross_entropy_with_logits(calibrated_logits77[cal_idx77],ty77[cal_idx77])); calibrated_metric77=calibration_metrics77(calibrated_logits77[test_idx77],ty77[test_idx77])
assert temperature77>1. and after_cal_nll77<before_cal_nll77
assert calibrated_metric77["nll"]<raw_metric77["nll"] and calibrated_metric77["brier"]<raw_metric77["brier"]
assert torch.equal(torch.argsort(raw_logits77),torch.argsort(calibrated_logits77))

## 5. 阈值是业务决策，不是固定 0.5

定义 false positive 成本 2、false negative 成本 5，使用独立 threshold split 穷举候选阈值，最小化每样本期望实现成本。阈值选择还应满足容量/precision/recall 约束；这里先做单目标成本。

test 只评估选定阈值，不再调参。

In [ ]:
def confusion77(prob,label,threshold):
    pred=prob>=threshold; label=np.asarray(label,bool); return {"tp":int(np.sum(pred&label)),"fp":int(np.sum(pred&~label)),"tn":int(np.sum(~pred&~label)),"fn":int(np.sum(~pred&label))}
def cost77(c): return (2*c["fp"]+5*c["fn"])/sum(c.values())
threshold_idx77=splits77["threshold"]; p_threshold77=torch.sigmoid(calibrated_logits77[threshold_idx77]).numpy(); y_threshold77=y77[threshold_idx77]
candidates77=np.linspace(.05,.95,181); costs77=np.array([cost77(confusion77(p_threshold77,y_threshold77,t)) for t in candidates77]); best_threshold77=float(candidates77[costs77.argmin()])
test_prob77=torch.sigmoid(calibrated_logits77[test_idx77]).numpy(); test_conf77=confusion77(test_prob77,y77[test_idx77],best_threshold77); default_conf77=confusion77(test_prob77,y77[test_idx77],.5)
assert .05<=best_threshold77<=.95 and cost77(test_conf77)<=cost77(default_conf77)+.05
assert sum(test_conf77.values())==len(test_idx77) and all(v>=0 for v in test_conf77.values())
assert cost77(confusion77(np.array([.9,.1]),np.array([1,0]),.5))==0.

## 6. 拒绝区间与 coverage-risk

对 `|p-threshold|` 小的样本交人工/更贵模型，剩余自动决策。报告 coverage 和已覆盖样本成本；扩大拒绝带通常降风险但增加人工量。不能只报自动样本准确率而隐藏大量拒绝。

这里在 threshold split 选一个固定宽度示例，再在 test 报告。

In [ ]:
def abstain_metrics77(prob,label,threshold,width):
    keep=np.abs(prob-threshold)>=width
    if not keep.any(): return {"coverage":0.,"cost":float("nan"),"kept":0}
    return {"coverage":float(keep.mean()),"cost":cost77(confusion77(prob[keep],np.asarray(label)[keep],threshold)),"kept":int(keep.sum())}
no_abstain77=abstain_metrics77(test_prob77,y77[test_idx77],best_threshold77,0.); abstain77=abstain_metrics77(test_prob77,y77[test_idx77],best_threshold77,.1)
assert 0<abstain77["coverage"]<1 and no_abstain77["coverage"]==1.
assert abstain77["kept"]<no_abstain77["kept"] and math.isfinite(abstain77["cost"])
assert abstain_metrics77(np.array([.5]),np.array([1]),.5,1.)["coverage"]==0.

## 7. 切片校准、漂移与再校准触发

总体 ECE 可能掩盖 group 差异。按来源计算 NLL/ECE/count，并检查后期时间窗口。再校准触发应基于足够 label 延迟后的稳定窗口，不能用未成熟标签。

切片小样本 ECE 方差很大，需同时显示 count 和置信区间。

In [ ]:
slice_metrics77={}
for g in (0,1):
    mask=group77[test_idx77]==g; slice_metrics77[g]=calibration_metrics77(calibrated_logits77[test_idx77][mask],ty77[test_idx77][mask])
assert all(sum(row[2] for row in slice_metrics77[g]["bins"])==int((group77[test_idx77]==g).sum()) for g in (0,1))
assert all(math.isfinite(slice_metrics77[g]["nll"]) for g in (0,1))
assert abs(slice_metrics77[0]["nll"]-slice_metrics77[1]["nll"])<.3
assert calibrated_metric77["ece"]<.12

## 8. 发布 wrapper 与完整决策合同

manifest 绑定 base state、特征顺序、temperature、业务成本、threshold、abstain width、四段时间 split 和评估。服务返回 probability、decision 与是否 abstain；拒绝 NaN/shape 错误。

更新 base model 后旧 temperature/threshold 全部失效，必须重新校准并新版本发布。

In [ ]:
def state_digest77(model):
    h=hashlib.sha256()
    for k,v in sorted(model.state_dict().items()): a=v.detach().numpy(); h.update(k.encode()); h.update(str(a.dtype).encode()); h.update(a.tobytes())
    return h.hexdigest()
manifest77={"artifact_id":"calibrated-classifier-v1","features":["x0","x1","x2"],"state":state_digest77(model77),"logit_scale_before_calibration":1.8,"temperature":temperature77,"threshold":best_threshold77,"abstain_width":.1,"cost":{"fp":2,"fn":5},"splits":{k:[int(v[0]),int(v[-1])+1] for k,v in splits77.items()}}
TRUST77=MappingProxyType({manifest77["artifact_id"]:sha77(canonical77(manifest77).encode())})
class PublishedClassifier77:
    def __init__(self,model,m): self._model=model.eval(); self.temperature=m["temperature"]; self.threshold=m["threshold"]; self.width=m["abstain_width"]
    @torch.no_grad()
    def predict(self,x):
        x=torch.as_tensor(x,dtype=torch.float32); logits=self._model(x)*1.8/self.temperature; p=torch.sigmoid(logits); abstain=(p-self.threshold).abs()<self.width
        return {"probability":p,"positive":p>=self.threshold,"abstain":abstain}
def load_classifier77(m,model):
    actual=copy.deepcopy(m); actual["state"]=state_digest77(model)
    if TRUST77.get(actual.get("artifact_id"))!=sha77(canonical77(actual).encode()): raise RuntimeError("untrusted_calibrator")
    return PublishedClassifier77(model,actual)
published77=load_classifier77(manifest77,model77); served77=published77.predict(x77[:3])
assert served77["probability"].shape==(3,) and torch.isfinite(served77["probability"]).all()
forged77=Logistic77()
try: load_classifier77(manifest77,forged77); raise AssertionError("forged classifier accepted")
except RuntimeError as e: assert str(e)=="untrusted_calibrator"
print({"temperature":round(temperature77,3),"raw_nll":round(raw_metric77["nll"],4),"cal_nll":round(calibrated_metric77["nll"],4),"threshold":round(best_threshold77,3),"test_cost":round(cost77(test_conf77),3)})

## 9. 失败模式、复杂度与来源

Temperature scaling 拟合成本线性于 calibration 样本；阈值扫描 `O(NT)`。常见错误：在 test 拟合温度/阈值、把 AUC 当校准、ECE 不报 bins/count、base 升级沿用旧温度、阈值固定 0.5、切片失衡和拒绝样本不计 coverage。

- Guo et al., [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html), ICML 2017。
- Niculescu-Mizil & Caruana, [Predicting Good Probabilities](https://www.cs.cornell.edu/~alexn/papers/calibration.icml05.crc.rev3.pdf)。
- Gneiting & Raftery, [Strictly Proper Scoring Rules](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jasa.pdf)。